# 최종 분석 전체 실행

종로구·마포구의 외식업 10종, 2023~2025년 분기 분석을 **01→08 순서로 각각 새 커널에서 실행**한다. 고정 입력 검증부터 데이터셋·회귀·RDF·분석 도구까지 다시 생성한다. 수집 노트북은 실행하지 않으며 API·인증키가 필요하지 않다.

**저장 위치:** 분석 자료 `dataset/analysis/final/`, 결과·읽기용 HTML `reports/20261006/`. 기존 분석과 원자료는 덮어쓰지 않는다. 이 노트북 실행 중 개별 분석 노트북을 동시에 실행하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, time
import nbformat
from nbclient import NotebookClient
from nbconvert import HTMLExporter
import pandas as pd
from IPython.display import display

ROOT = PROJECT_ROOT
FINAL = project_path('jupyter-notebook/20261006')
OUT = project_path('reports/20261006')
HTML = OUT / 'notebooks'
HTML.mkdir(parents=True, exist_ok=True)
for name in ['OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'OMP_NUM_THREADS']:
    os.environ[name] = '1'


## 1. 실행할 노트북 확인

번호가 중복되지 않는 최종본만 실행한다. 수집 도구와 archive의 이전 코드는 실행 목록에 포함하지 않는다.

In [ ]:
STEPS = [
 '01_입력검증과_준비.ipynb',
 '02_공연과_상권_연결.ipynb',
 '03_연결검증.ipynb',
 '04_외식업_데이터셋.ipynb',
 '05_외식업_고정효과분석.ipynb',
 '06_GEE_보조분석.ipynb',
 '07_RDF_지식그래프.ipynb',
 '08_분석도구.ipynb',
]
paths = [FINAL / 'notebooks' / name for name in STEPS]
assert all(p.is_file() for p in paths), '최종 노트북이 모두 있어야 한다.'
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
manifest_hashes = {p.relative_to(ROOT).as_posix():file_hash(p)
                   for p in [FINAL/'metadata'/name for name in ['kopis_analysis_sources.json','seoul_analysis_sources.json','validation_sources.json']]}
assert len(manifest_hashes) == 3, '서울시·KOPIS·검증자료 고정 명세가 필요하다.'
display(pd.DataFrame({'순서':range(1,len(paths)+1),'노트북':STEPS}))


## 2. 새 커널 실행과 결과 내보내기

각 노트북의 모든 셀을 위에서 아래로 실행하고 오류가 있으면 즉시 중단한다. 성공한 노트북은 출력과 함께 저장하고 HTML을 만든다. 실패한 실행은 `reports/20261006/failed/`에서 확인할 수 있다.

In [ ]:
def run_notebook(path):
    notebook = nbformat.read(path, as_version=4)
    nbformat.validate(notebook)
    source = '\n'.join(c.source for c in notebook.cells)
    started = time.perf_counter()
    try:
        NotebookClient(notebook, timeout=900, kernel_name='python3',
            resources={'metadata':{'path':str(path.parent)}}).execute()
    except Exception:
        failed = OUT / 'failed'
        failed.mkdir(exist_ok=True)
        nbformat.write(notebook, failed/path.name)
        raise
    nbformat.validate(notebook)
    nbformat.write(notebook, path)
    body, _ = HTMLExporter(template_name='lab').from_notebook_node(notebook)
    html_path = HTML / (path.stem+'.html')
    html_path.write_text(body, encoding='utf-8')
    return {'notebook':path.relative_to(ROOT).as_posix(), 'status':'passed',
        'seconds':round(time.perf_counter()-started,2),
        'code_cells':sum(c.cell_type=='code' for c in notebook.cells),
        'source_sha256':hashlib.sha256(source.encode('utf-8')).hexdigest(),
        'executed_sha256':file_hash(path),'html':html_path.relative_to(ROOT).as_posix()}


In [ ]:
receipt = {'started_at':datetime.now(timezone.utc).isoformat(),
           'status':'running','input_manifest_sha256':manifest_hashes,'steps':[]}
receipt_path = OUT / 'execution_manifest.json'
def save_receipt():
    receipt_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
save_receipt()
try:
    for index,path in enumerate(paths,1):
        print(f'[{index}/{len(paths)}] {path.name}',flush=True)
        receipt['steps'].append(run_notebook(path))
        save_receipt()
    assert all(file_hash(project_path(p))==digest for p,digest in manifest_hashes.items())
    receipt.update(status='completed',finished_at=datetime.now(timezone.utc).isoformat())
except Exception as error:
    receipt.update(status='failed',failed_notebook=path.name,error_type=type(error).__name__)
    raise
finally:
    save_receipt()
display(pd.DataFrame(receipt['steps'])[['notebook','status','seconds','code_cells']])


## 3. 완료 여부와 해석

모든 단계가 `passed`이면 최종본의 입력·실행 연결이 확인된 것이다. 통계적 유의성이나 인과효과를 보증하는 표식은 아니다. 점포는 일반·프랜차이즈·전체로 구분하고, 자료가 제공된 상권·업종의 분기 관련성으로 해석한다. 현재 경계의 과거 유효성, 선택적 매출 미제공, 상권 사이 공간 의존은 남은 한계다.

In [ ]:
assert receipt['status']=='completed' and len(receipt['steps'])==8
print('최종 분석 8개 노트북 실행 완료')
print('실행 기록:',receipt_path.relative_to(ROOT))
print('읽기용 HTML:',HTML.relative_to(ROOT))
